# MLflow 3.x — Introducción práctica

Este notebook es un recorrido por los conceptos fundamentales de MLflow usando un **modelo deliberadamente tonto** (siempre predice lo mismo) para que el foco esté en MLflow, no en el modelo.

Al terminar sabrás:
- Qué es un **Experiment** y un **Run**
- Cómo loggear **params**, **metrics** y **artifacts**
- Qué hace `autolog` por debajo
- Cómo registrar un modelo en el **Model Registry** y asignarle un alias
- Cómo cargar un modelo desde el registry por alias

> **Requisito:** el stack Docker debe estar levantado (`cd infra && docker compose up -d`). MLflow estará en http://localhost:5001

## 0. Setup — conectar al servidor MLflow

In [6]:
import mlflow
import mlflow.sklearn
import numpy as np
from mlflow import MlflowClient

TRACKING_URI = "http://localhost:5001"
mlflow.set_tracking_uri(TRACKING_URI)

print(f"MLflow version : {mlflow.__version__}")
print(f"Tracking URI   : {mlflow.get_tracking_uri()}")

MLflow version : 3.11.1
Tracking URI   : http://localhost:5001


---
## 1. Conceptos fundamentales

MLflow organiza el trabajo en cuatro niveles:

| Concepto | Qué es | Analogía |
|----------|--------|----------|
| **Tracking URI** | La dirección del servidor MLflow | La dirección del laboratorio |
| **Experiment** | Un contenedor lógico de runs relacionados | Una carpeta de proyecto |
| **Run** | Una ejecución concreta: entrenas una vez, es un run | Una entrada del cuaderno de laboratorio |
| **Artifact** | Ficheros guardados: el modelo, plots, CSVs | Los adjuntos de esa entrada |

Dentro de cada **Run** se loggean tres tipos de datos:
- **Params** — inputs fijos: hiperparámetros, rutas, configuración. No cambian durante el run.
- **Metrics** — outputs numéricos: accuracy, loss, AUC. Pueden evolucionar en el tiempo (por epoch).
- **Tags** — metadata libre en formato clave/valor: autor, tipo de modelo, dataset usado.

---
## 2. El modelo tonto y los datos ficticios

Usamos `DummyClassifier` de scikit-learn — un modelo que ignora completamente los datos de entrada y siempre predice lo mismo. No sirve para nada en producción, pero es perfecto para aprender MLflow porque:

- Se entrena en microsegundos
- No tiene hiperparámetros complejos que distraigan
- Es sklearn-compatible, así que funciona con todas las APIs de MLflow

Los datos son 10 filas inventadas con 2 features y un target binario.

In [7]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# Datos ficticios: 10 clientes, 2 features (edad, meses_contrato), target: churn
X = np.array([
    [25, 1], [45, 24], [30, 3], [55, 36], [22, 2],
    [40, 12], [28, 1], [60, 48], [35, 6], [50, 1]
])
y = np.array([1, 0, 1, 0, 1, 0, 1, 0, 0, 1])  # 1 = hace churn

print(f"Dataset: {X.shape[0]} filas, {X.shape[1]} features")
print(f"Churn rate: {y.mean():.0%}")

Dataset: 10 filas, 2 features
Churn rate: 50%


---
## 3. Crear un Experiment

Todos los runs se agrupan bajo un Experiment. Si no existe, MLflow lo crea automáticamente.

Sin `set_experiment`, los runs van al experimento `Default` — funciona, pero es desorden.

In [8]:
experiment = mlflow.set_experiment("mlflow-intro")

print(f"Experiment name : {experiment.name}")
print(f"Experiment ID   : {experiment.experiment_id}")
print(f"Artifact store  : {experiment.artifact_location}")

Experiment name : mlflow-intro
Experiment ID   : 1
Artifact store  : mlflow-artifacts:/1


Abre http://localhost:5001 — ya deberías ver el experimento `mlflow-intro` en el panel izquierdo. Todavía no tiene runs.

---
## 4. Primer Run — logging manual

`mlflow.start_run()` abre un run. El bloque `with` lo cierra automáticamente al salir (y lo marca como `FINISHED`).

Si no usas `with`, tienes que llamar `mlflow.end_run()` a mano — si no, el run queda en estado `RUNNING` para siempre.

In [9]:
with mlflow.start_run(run_name="dummy-siempre-uno") as run:

    # --- PARAMS: inputs del experimento ---
    mlflow.log_param("strategy", "constant")
    mlflow.log_param("constant", 1)
    mlflow.log_param("modelo", "DummyClassifier")

    # --- MODELO ---
    model = DummyClassifier(strategy="constant", constant=1)
    model.fit(X, y)
    preds = model.predict(X)

    # --- METRICS: outputs del experimento ---
    mlflow.log_metric("accuracy", accuracy_score(y, preds))
    mlflow.log_metric("f1", f1_score(y, preds, zero_division=0))

    # --- TAGS: metadata libre ---
    mlflow.set_tag("autor", "clase")
    mlflow.set_tag("nota", "modelo trivial para la demo")

    # --- MODELO COMO ARTIFACT ---
    mlflow.sklearn.log_model(model, artifact_path="model")

    run_id_v1 = run.info.run_id

print(f"\nRun ID  : {run_id_v1}")
print(f"Status  : {run.info.status}")
print(f"UI      : {TRACKING_URI}/#/experiments/{experiment.experiment_id}/runs/{run_id_v1}")

2026/04/28 18:38:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/28 18:38:04 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/28 18:38:06 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run dummy-siempre-uno at: http://localhost:5001/#/experiments/1/runs/88af1969d94d470f86161f8709a47879
🧪 View experiment at: http://localhost:5001/#/experiments/1

Run ID  : 88af1969d94d470f86161f8709a47879
Status  : RUNNING
UI      : http://localhost:5001/#/experiments/1/runs/88af1969d94d470f86161f8709a47879


**Ir a la UI ahora.** Deberías ver el run `dummy-siempre-uno` con:
- Pestaña **Parameters**: strategy, constant, modelo
- Pestaña **Metrics**: accuracy y f1
- Pestaña **Tags**: autor, nota + los tags de sistema que MLflow añade solo
- Pestaña **Artifacts**: la carpeta `model/` con el modelo serializado, `MLmodel`, `requirements.txt`

> ¿Por qué la accuracy es alta con un modelo tonto? Porque el dataset tiene 50% de churn. Siempre predecir 1 acierta exactamente la mitad. En el dataset real de churn (~26%), siempre predecir 0 daría 74% de accuracy — el modelo tonto parecería bueno. Por eso no usamos accuracy.

---
## 5. Inspeccionar un run con MlflowClient

La UI es bonita para explorar a mano, pero en código se usa `MlflowClient` para acceder a los datos de forma programática.

In [10]:
client = MlflowClient()
run_data = client.get_run(run_id_v1)

print("Params  :", run_data.data.params)
print("Metrics :", run_data.data.metrics)
print("Tags    :", {k: v for k, v in run_data.data.tags.items() if not k.startswith("mlflow.")})

Params  : {'strategy': 'constant', 'constant': '1', 'modelo': 'DummyClassifier'}
Metrics : {'accuracy': 0.5, 'f1': 0.6666666666666666}
Tags    : {'autor': 'clase', 'nota': 'modelo trivial para la demo'}


---
## 6. Segundo Run — comparar en la UI

La razón principal para usar MLflow es poder comparar runs. Vamos a entrenar un segundo modelo ligeramente diferente.

In [11]:
with mlflow.start_run(run_name="dummy-mayoria") as run:

    mlflow.log_param("strategy", "most_frequent")
    mlflow.log_param("modelo", "DummyClassifier")

    model2 = DummyClassifier(strategy="most_frequent")
    model2.fit(X, y)
    preds2 = model2.predict(X)

    mlflow.log_metric("accuracy", accuracy_score(y, preds2))
    mlflow.log_metric("f1", f1_score(y, preds2, zero_division=0))

    mlflow.sklearn.log_model(model2, artifact_path="model")
    run_id_v2 = run.info.run_id

print(f"Run ID v2: {run_id_v2}")

2026/04/28 18:38:08 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/28 18:38:08 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/28 18:38:10 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run dummy-mayoria at: http://localhost:5001/#/experiments/1/runs/88d9ba7b6fed499fb41cc26a8ce51610
🧪 View experiment at: http://localhost:5001/#/experiments/1
Run ID v2: 88d9ba7b6fed499fb41cc26a8ce51610


**En la UI:** selecciona los dos runs con el checkbox → botón **Compare**.

Verás una tabla comparativa de params y metrics, y un **Parallel Coordinates Plot** que muestra visualmente cómo cada hiperparámetro afecta a cada métrica. Con dos modelos tontos no hay mucho que ver — pero con runs reales este gráfico se vuelve muy útil.

Sin MLflow: tendrías dos scripts, dos ficheros `.pkl` con nombres como `model_v1_final.pkl` y `model_v2_prueba.pkl`, y ninguna forma de saber qué params usó cada uno.

---
## 7. Metrics con steps — evolución temporal

Las métricas pueden tener un parámetro `step` para representar evolución a lo largo del entrenamiento (epochs, iteraciones). Esto activa el gráfico de línea en la UI.

In [12]:
with mlflow.start_run(run_name="con-steps"):
    mlflow.log_param("epochs", 10)

    # Simular una curva de entrenamiento
    for epoch in range(10):
        train_loss = 1.0 / (epoch + 1)
        val_loss = 1.2 / (epoch + 1) + np.random.uniform(0, 0.05)
        mlflow.log_metric("train_loss", train_loss, step=epoch)
        mlflow.log_metric("val_loss",   val_loss,   step=epoch)

print("Run completado — abre la UI y haz clic en 'train_loss' para ver el gráfico")

🏃 View run con-steps at: http://localhost:5001/#/experiments/1/runs/4fbb92fe79fc44ecb70328b317fafa3c
🧪 View experiment at: http://localhost:5001/#/experiments/1
Run completado — abre la UI y haz clic en 'train_loss' para ver el gráfico


---
## 8. Artifacts — guardar ficheros

Un artifact es cualquier fichero que quieras guardar junto al run: plots, CSVs, JSONs de configuración, informes HTML. Se almacenan en MinIO (nuestro S3 local).

In [13]:
import json
import matplotlib.pyplot as plt

with mlflow.start_run(run_name="con-artifacts"):

    # 1. Loggear un dict directamente (sin crear fichero en disco)
    mlflow.log_dict(
        {"threshold": 0.5, "features": ["edad", "meses_contrato"]},
        "config.json"
    )

    # 2. Crear un plot y loggearlo
    fig, ax = plt.subplots(figsize=(5, 3))
    ax.bar(["No churn", "Churn"], [(y == 0).sum(), (y == 1).sum()],
           color=["#4CAF50", "#F44336"])
    ax.set_title("Distribución del target")
    plt.tight_layout()
    mlflow.log_figure(fig, "distribucion_target.png")
    plt.close()

    # 3. Crear un fichero local y loggearlo
    with open("resumen.txt", "w") as f:
        f.write("Modelo: DummyClassifier\n")
        f.write(f"Filas: {len(X)}\n")
    mlflow.log_artifact("resumen.txt")

    mlflow.log_metric("accuracy", 0.5)

print("Run completado — explora la pestaña Artifacts en la UI")

🏃 View run con-artifacts at: http://localhost:5001/#/experiments/1/runs/6fa23fe1e5de479ea0427f0a34e681e3
🧪 View experiment at: http://localhost:5001/#/experiments/1
Run completado — explora la pestaña Artifacts en la UI


---
## 9. Autolog — una línea para capturar todo

Con logging manual escribes cada `log_param` y `log_metric` a mano. `autolog` se engancha al `fit()` de scikit-learn y captura automáticamente:
- Todos los hiperparámetros del modelo
- Métricas de entrenamiento (accuracy, F1, etc.)
- El modelo serializado como artifact
- El input example y la firma del modelo
- Las dependencias del entorno (`requirements.txt`)

Basta con llamarlo una vez antes del `start_run`.

In [14]:
mlflow.sklearn.autolog(log_input_examples=True, log_model_signatures=True)

with mlflow.start_run(run_name="con-autolog") as run:
    model3 = DummyClassifier(strategy="stratified", random_state=42)
    model3.fit(X, y)
    run_id_autolog = run.info.run_id

print(f"Run ID: {run_id_autolog}")
print("Explora la UI — no escribimos ningún log_param ni log_metric, pero están todos")

2026/04/28 18:38:13 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/28 18:38:14 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run con-autolog at: http://localhost:5001/#/experiments/1/runs/b9a54c3b1df1477fb9c39475129b3e8e
🧪 View experiment at: http://localhost:5001/#/experiments/1
Run ID: b9a54c3b1df1477fb9c39475129b3e8e
Explora la UI — no escribimos ningún log_param ni log_metric, pero están todos


In [15]:
# Ver qué capturó autolog
run_autolog = client.get_run(run_id_autolog)
print("Params capturados por autolog:")
for k, v in run_autolog.data.params.items():
    print(f"  {k}: {v}")

print("\nMetrics capturadas por autolog:")
for k, v in run_autolog.data.metrics.items():
    print(f"  {k}: {v:.4f}")

Params capturados por autolog:
  constant: None
  random_state: 42
  strategy: stratified

Metrics capturadas por autolog:
  training_precision_score: 0.7083
  training_recall_score: 0.7000
  training_f1_score: 0.6970
  training_accuracy_score: 0.7000
  training_log_loss: 10.8131
  training_roc_auc: 0.7000
  training_score: 0.7000


---
## 10. Model Registry — del experimento al catálogo

Hasta aquí los runs viven en el área de **Experiments**: están rastreados, pero son efímeros en concepto — son registros de lo que probaste.

El **Model Registry** es diferente: es un catálogo de modelos que se consideran listos para usarse en algún entorno. Cuando registras un modelo, le das:
- Un **nombre** que identifica qué hace (`churn-model`, `fraud-detector`)
- Un **número de versión** automático (1, 2, 3...)
- Opcionalmente, un **alias** que el código de producción puede usar

### Registrar un modelo

El modelo tiene que estar loggeado como artifact en un run. Lo referenciamos con `runs:/<run_id>/<artifact_path>`.

In [16]:
MODEL_NAME = "intro-dummy-model"

# Registramos el primer modelo (run_id_v1) como versión 1
result = mlflow.register_model(
    model_uri=f"runs:/{run_id_v1}/model",
    name=MODEL_NAME,
)

version_1 = result.version
print(f"Modelo registrado: {MODEL_NAME} — versión {version_1}")
print(f"URI en el registry: models:/{MODEL_NAME}/{version_1}")

Successfully registered model 'intro-dummy-model'.
2026/04/28 18:38:16 WARNING mlflow.tracking._model_registry.fluent: Run with id 88af1969d94d470f86161f8709a47879 has no artifacts at artifact path 'model', registering model based on models:/m-e9d047ec0bbd424089cf08095bdb0335 instead
2026/04/28 18:38:16 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: intro-dummy-model, version 1


Modelo registrado: intro-dummy-model — versión 1
URI en el registry: models:/intro-dummy-model/1


Created version '1' of model 'intro-dummy-model'.


In [17]:
# Registramos el segundo modelo como versión 2
result2 = mlflow.register_model(
    model_uri=f"runs:/{run_id_v2}/model",
    name=MODEL_NAME,
)
version_2 = result2.version
print(f"Versión 2 registrada: {MODEL_NAME} v{version_2}")

Registered model 'intro-dummy-model' already exists. Creating a new version of this model...
2026/04/28 18:38:16 WARNING mlflow.tracking._model_registry.fluent: Run with id 88d9ba7b6fed499fb41cc26a8ce51610 has no artifacts at artifact path 'model', registering model based on models:/m-e6574ad0a16848b7919e0304da473abc instead
2026/04/28 18:38:16 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: intro-dummy-model, version 2


Versión 2 registrada: intro-dummy-model v2


Created version '2' of model 'intro-dummy-model'.


**En la UI:** ve a **Models** (menú superior). Verás `intro-dummy-model` con dos versiones. Cada versión apunta al run original y muestra su linaje completo.

### Aliases — apuntar a versiones sin hardcodear números

El problema con usar el número de versión en código: si tienes `models:/intro-dummy-model/1` hardcodeado en producción y quieres actualizar al modelo 3, tienes que cambiar el código y hacer un deploy.

Con aliases, el código siempre carga `models:/intro-dummy-model@champion`. Cuando quieres actualizar, solo cambias a qué versión apunta el alias `champion` — el código no toca.

In [18]:
# Asignar alias @champion a la versión 1
client.set_registered_model_alias(
    name=MODEL_NAME,
    alias="champion",
    version=version_1,
)

# Asignar alias @challenger a la versión 2
client.set_registered_model_alias(
    name=MODEL_NAME,
    alias="challenger",
    version=version_2,
)

print(f"@champion  → versión {version_1}")
print(f"@challenger → versión {version_2}")

@champion  → versión 1
@challenger → versión 2


In [19]:
# Verificar: qué versión tiene cada alias
champion_info = client.get_model_version_by_alias(MODEL_NAME, "champion")
print(f"@champion  → versión {champion_info.version} (run {champion_info.run_id[:8]}...)")

challenger_info = client.get_model_version_by_alias(MODEL_NAME, "challenger")
print(f"@challenger → versión {challenger_info.version} (run {challenger_info.run_id[:8]}...)")

@champion  → versión 1 (run 88af1969...)
@challenger → versión 2 (run 88d9ba7b...)


---
## 11. Cargar el modelo por alias

El código de inferencia carga siempre el `@champion`. Cuando promovemos un nuevo modelo, el alias cambia de versión — el código no cambia.

In [20]:
# Cargar el @champion desde el registry
champion_model = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}@champion")

predictions = champion_model.predict(X)
print(f"Modelo cargado: {type(champion_model).__name__}")
print(f"Predicciones   : {predictions}")
print(f"Target real    : {y}")

Modelo cargado: DummyClassifier
Predicciones   : [1 1 1 1 1 1 1 1 1 1]
Target real    : [1 0 1 0 1 0 1 0 0 1]


In [21]:
# Simular una promoción: el challenger pasa a ser champion
client.set_registered_model_alias(
    name=MODEL_NAME,
    alias="champion",
    version=version_2,  # ahora el alias apunta a la versión 2
)

# El mismo código de carga funciona sin cambios
nuevo_champion = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}@champion")
nueva_version = client.get_model_version_by_alias(MODEL_NAME, "champion").version
print(f"@champion ahora es la versión {nueva_version}: {type(nuevo_champion).__name__}")
print("El código de producción no cambió — solo el alias")

@champion ahora es la versión 2: DummyClassifier
El código de producción no cambió — solo el alias


---
## 12. Resumen

Lo que hemos visto:

| Qué hicimos | Cómo | Para qué |
|-------------|------|----------|
| Conectar al servidor | `mlflow.set_tracking_uri()` | Apuntar al backend remoto |
| Crear un experiment | `mlflow.set_experiment()` | Agrupar runs relacionados |
| Abrir un run | `with mlflow.start_run()` | Contenedor de una ejecución |
| Loggear params | `mlflow.log_param()` | Registrar hiperparámetros |
| Loggear métricas | `mlflow.log_metric()` | Registrar resultados |
| Loggear artifacts | `mlflow.log_artifact()`, `log_figure()`, `log_dict()` | Guardar ficheros |
| Loggear el modelo | `mlflow.sklearn.log_model()` | Serializar y guardar el modelo |
| Automatizar todo | `mlflow.sklearn.autolog()` | Capturar todo sin código extra |
| Registrar un modelo | `mlflow.register_model()` | Promover al catálogo de modelos |
| Asignar un alias | `client.set_registered_model_alias()` | Etiquetar versiones (champion, challenger) |
| Cargar por alias | `mlflow.sklearn.load_model("models:/name@alias")` | Inferencia desacoplada de la versión |

### Próximo paso

En el siguiente bloque usaremos exactamente estas mismas APIs pero con el dataset real de churn y modelos reales (Logistic Regression → Random Forest → XGBoost). La única diferencia es que el modelo tendrá algo útil que aprender.